# 27. Collapse double-clicks into one click

**Track:** sql | **Difficulty:** medium | **Tags:** deduplication, lag, time window, window functions

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("027-collapse-double-clicks")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "clicks": """
UEFSMRUEFeArFY4WTBW8BRUAEgAA8BUM4l8BAAEBAN8NCAQQYAkQAPMNEAAvDRAA9g0QALsNCADNDQgAwQ0gAJ0NEADFDQgAgg0Y
AOkNEABBDRAArw0QAFMNEACrDQgAgA0IAKENCAAXDQgAlQ0wABQNEAAiDQgAnA0YALwNCACbDRgA7Q0QAJQNCADEDQgAAw0gALYN
CABMDQgA2g0IAPkNKAC0DRAADw0IAMINGABDDRAAUg0IAAENCAAGDQgAwA0oAJENEAALDQgAzg0YAPUNCAAcDRgA5g0IALcNCAAu
DQgA1w0IAA4NCACVDQgAzA1AAOwNCACPDRgArg0IAF8NCADZDSAAnQ0QAPQNEAAWDRAA+A0QABUNEACHDQgAag0IAJMNIABzDRAA
UA0IAO4NGACaDQgAvg0IADENIADTDQgAMg0IAFoNCADBDSgAiQ0QAJoNCABGDQgA0A0gAD4NEADUDQgAug0YAIoNEACIDQgA4g0I
AFQNCAC3DSgAPQ0QAJENEACDDRAA3Q0QANUNEACtDRAAlg0IALsNGADmDRAAsA0QAJANCAB9DQgALQ0IAFkNCABvDQgAGw0IABoN
CAC9DUgAnw0IAF0NGADFDQgAwA0IALwNCADGDSgABw0QAN4NEADNDRAAOg0IAG4NCADDDQgAsw0oAMgNEAAlDQgAeg0IABMNCAAZ
DQgAjQ0IAKQNOABCDRAAVg0IAIENCACiDQgAhQ0IALkNMADoDQgAZg0YAPwNEADxDQgAbQ0YAH4NCABwDQgAfw0IACENCABYDQgA
4w0IALUNQAAoDRAAuQ0IADUNCACmDQgAPw0IAOwNCADLDQgArw0IANUNSACXDQgANA0YAKANCABgDQgAcg0IACMNCADYDTAAdQ0Q
AO8NEABRDRAAqg0IAKwNGADKDQgADA0YAFsNCADHDRgAmA0QACQNCACoDQgARQ0IAOQNCAAqDQgA2g04ALENEADKDQgAlA0IAOsN
IABpDRAAlg0IADwNCADlDSAAYw0QAN0NCAAnDQgAnw0IAB8NCACyDTAAjg0QANsNCABPDQgAYg0IAGwNCADrDQgApg04AKgNCACi
DQgArQ0gAEANCAC4DRgA8A0IANINCADGDSAA3g0IAP4NGADhDQgAqQ0IAEsNIAC/DQgA9w0YAHwNEABkDQgA5w0YAMkNEADtDQgA
BA0IAN8NCACcDQgASQ0IALgNCABVDQgAcQ0IADcNCAA4DQgAkw0IAK4NaABEDRAA0A0IAMcNCAC+DQgAmQ0IABINCAAFDQgAug0I
AKkNCAApDQgAzg0IAOQNYABHDRAAHQ0IACYNCAAzDQgAMA0IANcNMACjDRAA3A0IALYNGAAsDRAA4w0QAKUNCAB3DRgAow0QANgN
EACMDQgAtQ0IAGsNCABeDQgAhg0IANENCACgDUAAew0QACsNCACnDRgAAg0QAJINCAB4DQgAyA0gAJcNEADWDQgAdg0IAIQNCACx
DSgAyw0IAJ4NGABoDQgAxA0IAKsNIAAYDRAAsw0IAJkNGAA2DRAACg0IANINCACYDSAA1A0IAJ4NCADJDQgA6g0IABENMADnDQgA
0w0YANsNCADPDRgAkg0QANwNCAA5DRgACA0IAP8NGADlDRAA+w0QAKwNEAC0DRAA+g0IALINGADRDRAADQ0QADsNCACqDRgAmw0I
AMINGAD9DRAAvw0IAOoNGADPDRAA8g0IAB4NGABcDQgAsA0YAEgNEACnDQgATQ0IAOENCABKDQgA7g0IANkNCAB0DQgAAA0IAOkN
CAChDVgAIA0QAGENCADMDQgAiw0IAMMNKADgDRAAZQ0IAOANGAC9DRAAZw0IAKUNCADoDQgATg0IAHkNCABXDQgA1g1APKRgAQAA
AAAACWABAAAAAAAVABWqBhW0BiwVvAUVEBUGFQYcGAjuYAEAAAAAABgIkV8BAAAAAAAWACgI7mABAAAAAAAYCJFfAQAAAAAAEREA
AACVA/SUAQMAAAC8BQEJWQACCBhAoICBAwgSKFjAoIGDBxAiSJhAoYKFCxgyaNjAoYOHDyBCiBhBooSJEyhSqFjBooWLFzBiyJhB
o4aNGzhy6NjBo4ePH0CCCBlCpIiRI0iSKFnCpImTJ1CiSJlCpYqVK1iyaNnCpYuXL2DCiBlDpoyZM2jSqFnDpo2bN3DiyJlDp46d
O3jy6NnDp4+fP4ACCRpEqJChQ4gSKVrEqJGjR5AiSZpEqZKlS5gyadrEqZOnT6BCiRpFqpSpU6hSqVrFqpWrV7BiyZpFq5atW7hy
6drFq5evX8CCCRtGrJixY8iSKVvGrJmzZ9CiSZtGrZq1a9iyadvGrZu3b+DCiRtHrpy5c+jSqVvHrp27d/DiyZtHr569e/jy6dvH
r5+/fwADChxIsKDBgwgTKlzIsKHDhxAjSpxIsaLFixgzatzIsaPHjyBDihxJsqTJkyhTqlzJsqXLlzBjypxJs6bNmzhz6tzJs6fP
n0CDCh1KtKjRo0iTKl3KtKnTp1CjSp1KtarVq1izat3KtSsAABUEFfAEFdYCTBVOFQASAAC4AgQRAAkBABMJBwQAEg0IABANCAAZ
DQgACA0IAA4NCAAgDQgABw0IAAoNCAAFDQgAGw0IAAkNCAAiDQgAIw0IABgNCAQdAAkBACEJBwQAAg0IAAwNCAAcDQgAFg0IAAsN
CAAlDQgABg0IAA0NCAAPDQgAFA0IAB4NCAABDQgAFw0IACQNCAAfDQgAAw0IABUNCAAaDQgABA0IPCcAAAAAAAAAJgAAAAAAAAAV
ABWiBBWsBCwVvAUVEBUGFQYcGAgnAAAAAAAAABgIAQAAAAAAAAAWACgIJwAAAAAAAAAYCAEAAAAAAAAAEREAAACRAvQQAQMAAAC8
BQEGWUAgDERhHEgCKAGxMI2jPAlEJEb0JMCjSNNACQdUPdZBEUcxXEqFCUgGaI1TbAMkQBqXdFVhGFXkOcemcYcXZRvyMaAXRseU
gWIRURThSIKQJVKEiIBViIMUAdToQIJijIxCIUZACoMkOQ7BfCCncKJXTggQcgpZEUX3jE1WEA6gQSJ0VWHUIEHXQAA5eY21OFNV
KaWxgBsZBmEzPU8oIY9GClhZYBGnMAwnSBk0NaImjGJUNQBnWYRUUBNYAYZiTBqlcNGUaET2UFuhTV20ZBIThSOVccSUAUkRDFd4
TOAhEM8lNIsQhR4UgtFjmMdgSQsnSp9UTt1jLhY5aQdGPiIjmiTFABQBABUEFUoVTEwVCBUAEgAAJUQEAAAAbGlrZQUAAABzaGFy
ZQkBCTh1YnNjcmliZQMAAABidXkVABXCARXIASwVvAUVEBUGFQYcNgAoCXN1YnNjcmliZRgDYnV5EREAAABh8GADAAAAvAUBAllk
i4/WMvqItJlIwQqTvEKYKnZkPFr3GVQxX/X6KKSB3TtZFNxxy1vvKPaxCw9OBF4Zh5CB75r/8sYp78mwzQdVB4gXfdIrd4svhMCS
d7ZatrFLCP/cO5gPFQQV0CsVmiRMFboFFQASAADoFSxoJhk2ly8GAICrVQIBCCzg1voMmC8GAOjHx3QBEAygDTBjARAMEDy1qQEQ
LAh7HS+WLwYAYGOQewUISKdZCpsvBgBo3UCplS8GAHhvO2YBGCyAFLL9mS8GAKBWClUBOAzgWrO7AUgMwGlp9gEoDIC8iRoBIBC4
Rl/HmgEoCKgA9gEQDIAblKUBEAwIa8wpATAMQCCxVQEwDJA0RB8BEAz4wz5LAQgIACwUBXAIGOxkBZAMuAc6hgE4DBjpvmMBcAjY
E6gFOAiApCsFkAyIS2nNARgMwEOM7wEoDJjC9tABSAyY+f95BcAIxvK7ASAMWLuX6AEgCKihuCUIDMhW3V8B0AxoJn3GATAMkEwF
DgGoDMAS7cwBMAwARDfVAQgIgKh+BSgMiJ40SAFADPD5/PABGAh4PtQlKAhA86olQAzoXJ02AUgoAMHOg50vBgBADskFkAgQ5RAl
aAxAwSVxAZAIyM5/BYAMABERbQFQDDiXT3kBiAxQrb1ZAVgMaLwuRgEYCOg2iiVADNilPS8FmAjK+NIBKAxY11uQASAMaDyajwEw
CPB3yCVADEB/LrMBEAhAwYkFEAyQrKAqASgM8MY8fgFACIBsayU4DEC+IZ0BEAwQaijVAbAMgLdaaQE4DADp7J8hkAyYZIpDAWgM
EAL2fgEgDOAHFG8BuAxASwmEARAMUOksJQFACPjqwyUYDEBCCy8BaAzgiul7AQgMSIrQzwEoDOC/CHwBSAywftmmBRAEHiUFSAyI
CCApARgIGEZDBTgMgE6sLQE4DEBzDYAlOAhhkxoBYAywX90PASgIwOi5BUAMgPUjNwGgDACsEQQBMAwo7xrjASAMYKSlcAGgDPDZ
/O8BIAzA3dlxAQgMaAmrAwEYDEAcdzcB4AzQd4LYATgIgDUjJcgMQNWO4QFoDNDw9GIBoAiASAcF2AyoWiCGARgMgLLjNAEYCNhr
oAUIDIiCIzMBcAiAKnJFoAzoKognASgIoAHiZSgISHE5BQgM6CbXBwF4CLCHbUWgDIBLlt8BgAxAO3j0AUAMOF7aOwEgDOggwJoB
YAxYNPGFAUgIwNS3BUAMQKuJ+wHADAjtth4BKAzACw5dASgM2Pk51AEoCAAqICXgCCDvnQWYDACiFjsB2Ayg1FXOATgMgAaOvQEw
CJj9vCVICOAZcGWwDFhnK6kBKAxA+YsfAQgMCHFJIQGQDMAX41QBoAyYeGxtAWAMiKaHwgEQDICYVm4BCAio1OIFoAxoh7vuASAM
GECTkAEIDNju1/UBCAgwhitluAyw0uMfARAQmLpZgJ1hAAifkA8BYAhAX6glYAzILi/wAXgMiD9EjwWgCDEksAEQCAi8/SngKEGa
ip0vBgAAx/U1AfgMgMBEzQEgDNjFnNABSAwAw8qDAeAIwG0fBUAIwA1dhVAMAPeaOgF4DNDg4pcBCAhYw3oFkAiQ3+FlMAzobjWp
ARgM2Py+bAHQDAAm6NwBgAgAd1VlYAiANOpFOAzI6vNzAWgMMLK8AQEgCJjyZ0UQCOindCXICKBFnEXgDPACuFkBIAhgzVEFwAgA
IduFQAgAr1Rl+Ah43ckF8Ay4z6XTBVAEApNFeAxIDaEzAdgMQF3LbAHYCBgbZKVADADqE34BoAxALEdyARgMqDRDowFgCND9kqXA
DPBS5WwBIAw4w56ZAUAMgGIMYQEgDADy5qEBMAzAMVZJARAICO+5pYAMIArLPwEYDLijRHoBMAhQ9RZl6AxQPbFgAUgI4FgJJdAI
AL17JXAISICdRTgMwKH81SFgDIDKMssBCAiYOH4pIAgXcbABYAyAb44VAdAMGLrybSFICJAYqmXgDCjmFRUBaAiYcL4FoAxIDZXK
ASAM2P7aDQEIDIAc79oBUAigqfOlmAiAtxBF6AyYrSiwASAM0FQC3AGYCEjvDkWADIBkDUsBGAyAK3MuAVgIQAjIBagMEH4l1AEY
DEAM05sBGAxQ21+GAegIcKqMBVgI4HkNRWAMaEeyGgFQDGC0y5EBCAjIkI9FMAxAIn2YAcgMaDlkUgE4DIihJPABmAhYza4lwAwQ
T/9XAVgIiM8rBdAMOL0dCAEQDJjA4nsBMAzgFj0cAUAIQGuPBYgIoFSWhWAMkLrYwAEgCMDjxiX4DEgK5UcBOAj4q2clwAggoAUF
oAjAIEQlqAwwMOteAUgM4FtBjAEIDMANPHgBCAjI35pFsAjAAFxlqAw4JaeOAUgMKMW0DyF4CJB52QVgCAiUMwVYCFg8miW4DKD3
+McB6AgAuy0FEAj49YDF+AzoKFIxAZgISDSnxfgMAObAhAEoDHB7Py4BCAx4Dy8pASAMUGs+WAFoDIDJAMIhEAzAgfTbASAIWDXY
BXAIANvORRgI8Loj5WgI2KV9pUAMgFQbyAEoDAAhpHMBoAgoWoclKAj4seupkAQMU2V4DPggWQQBaAj4w5BFoAjIEQ7F8AzojAeV
ARgMIOn6cgFICEASzUUYDADs/eUFiAiTLzEhGAjQLVgFwAxAbWONARgIoAKMJdAI4EbA5YgIYA5UBcAMAEpehwEgCGjIboUwCHDP
UKWoCOANrwWgCNjqUWVwDOjukhgBWCwAyguEnS8GAHjBzXwBwAwgKdriAQgI4I+3RQgMQND5UgFQCFjd8gUYCLCuqEVACECIdAV4
CBjjxkXoCAC4kWX4DNjPd8JBoAyI9G3HAdgMSE0XBQFQCEC/cAUYCPh1nYUICICfjQVwDMDuHwQBiAgwGTZl8AxAkBXbAXAMgG0L
oAEIDPjXrgohmAhYUpAFaAxIh7dDBVAIePeGAbgIGOf5EoAICMDEmBLQCQhwP0hFYAhA+mrFmAjAA3olcAjg+kRFgAjQJzKJsAj+
DNEBeAwAKMF/AUgIQJ3OBSAI2IjmZWgI0NW+RUAMaDFonSGYDMDKBMsB2AgANugFeAxA5tjGAaAIMAZ2BXAIsK7EpYAoIN0oNpsv
BgAQJC9FeAygkfdfAbAM0AkOnAEYCADKK2XgCADXqIUYCPiI4uUoDDCKvXIBaAwABfivITgMgB6J0gGgCMCWKkVgDJi6vh8BIAho
NcUl2DzQ5Wapmi8GALBbqfCXLwYAFQAVqgYVtAYsFbwFFRAVBhUGHBgI2Ijmip0vBgAYCID1IzeVLwYAFgAoCNiI5oqdLwYAGAiA
9SM3lS8GABERAAAAlQP0lAEDAAAAvAUBCVkAAggYQKCAgQMIEihYwKCBgwcQIkiYQKGChQsYMmjYwKGDhw8gQogYQaKEiRMoUqhY
waKFixcwYsiYQaOGjRs4cujYwaOHjx9AgggZQqSIkSNIkihZwqSJkydQokiZQqWKlStYsmjZwqWLly9gwogZQ6aMmTNo0qhZw6aN
mzdw4siZQ6eOnTt48ujZw6ePnz+AAgkaRKiQoUOIEilaxKiRo0eQIkmaRKmSpUuYMmnaxKmTp0+gQokaRaqUqVOoUqlaxaqVq1ew
YsmaRauWrVu4cunaxauXr1/AggkbRqyYsWPIkilbxqyZs2fQokmbRq2atWvYsmnbxq2bt2/gwokbR66cuXPo0qlbx66du3fw4smb
R6+evXv48unbx6+fv38AAwocSLCgwYMIEypcyLChw4cQI0qcSLGixYsYM2rcyLGjx48gQ4ocSbKkyZMoU6pcybKly5cwY8qcSbOm
zZs4c+rcybOnz59AgwodSrSo0aNIkypdyrSp06dQo0qdSrWqVaVXsWbVupUrAAAVBBlcNQAYBnNjaGVtYRUIABUEJQIYCGNsaWNr
X2lkABUEJQIYB3VzZXJfaWQAFQwlAhgGYnV0dG9uJQBMHAAAABUEJQIYCmNsaWNrZWRfYXQlFEyMEhwsAAAAAAAWvAUZHBlMJgAc
FQQZNQAGEBkYCGNsaWNrX2lkFQIWvAUWsDMW6B0muBYmCBwYCO5gAQAAAAAAGAiRXwEAAAAAABYAKAjuYAEAAAAAABgIkV8BAAAA
AAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAvAUAAAAmABwVBBk1AAYQGRgHdXNlcl9pZBUCFrwFFrYKFqYIJuYgJvAdHBgIJwAA
AAAAAAAYCAEAAAAAAAAAFgAoCCcAAAAAAAAAGAgBAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgC8BQAAACYAHBUMGTUA
BhAZGAZidXR0b24VAha8BRb8AhaEAyb+JiaWJhw2ACgJc3Vic2NyaWJlGANidXkREQAZLBUEFQAVAgAVABUQFQIAPBa6HBkGGSYA
vAUAAAAmABwVBBk1AAYQGRgKY2xpY2tlZF9hdBUCFrwFFqAzFvQrJtZNJpopHBgI2Ijmip0vBgAYCID1IzeVLwYAFgAoCNiI5oqd
LwYAGAiA9SM3lS8GABERABksFQQVABUCABUAFRAVAgA8KQYZJgC8BQAAABaCdBa8BSYIFoZVABkcGAxBUlJPVzpzY2hlbWEYjAMv
Ly8vL3lBQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBUUFBQUMw
QUFBQWNBQUFBRUFBQUFBRUFBQUFiUC8vL3dBQUFRb1FBQUFBSkFBQUFBUUFBQUFBQUFBQUNnQUFBR05zYVdOclpXUmZZWFFBQUFB
QUJnQUlBQVlBQmdBQUFBQUFBZ0NrLy8vL0FBQUJCUkFBQUFBY0FBQUFCQUFBQUFBQUFBQUdBQUFBWW5WMGRHOXVBQUFFQUFRQUJB
QUFBTkQvLy84QUFBRUNFQUFBQUJnQUFBQUVBQUFBQUFBQUFBY0FBQUIxYzJWeVgybGtBTHovLy84QUFBQUJRQUFBQUJBQUZBQUlB
QVlBQndBTUFBQUFFQUFRQUFBQUFBQUJBaEFBQUFBa0FBQUFCQUFBQUFBQUFBQUlBQUFBWTJ4cFkydGZhV1FBQUFBQUNBQU1BQWdB
QndBSUFBQUFBQUFBQVVBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlMHAAAHAAAHAAAHAAAAOYDAABQ
QVIx
""",
}

_data_dir = pathlib.Path("data") / "027-collapse-double-clicks"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

clicks = spark.read.parquet((_data_dir / "clicks.parquet").resolve().as_posix())
clicks.createOrReplaceTempView("clicks")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'button': '06e395c0f9a8002207371eef207919b2c076dd0007cad4012f8d69d7e4047906',
                         'click_id': 'a3643db5fb3ed4ba26df6469a6952fc0f5f422315c282e42036c9c6eb823e838',
                         'clicked_at': '652d5fb030a497ff0ffa5d7aef96090460f9e2f7e222186ab6b8eeb930eef6ad',
                         'user_id': '5c539522a0e76f7335dbd5e1a1c1560e10b5838ae9b616e3b15e4fe7093b1e4e'},
 'columns': [['button', 'string'],
             ['click_id', 'bigint'],
             ['clicked_at', 'timestamp_ntz'],
             ['user_id', 'bigint']],
 'fingerprint': '946ffa8e284cdf6596fdef616e9279844b7c4f7e2fe2fb8d9c24e396caebddd5',
 'order_matters': False,
 'row_count': 188}

check = make_check(EXPECTED)

## The situation

The Buy button on a slow page gets clicked two or three times by impatient users, and
every click is recorded. Downstream counts of "button clicks" are inflated.

The product team has settled on a rule for what counts as an accidental repeat, and
needs the click log with the repeats removed.

## Input tables

### `clicks`

One row per recorded click.

| Column | Type | Description |
|---|---|---|
| `click_id` | BIGINT | Unique. A click recorded later has a higher id. |
| `user_id` | BIGINT | The user who clicked. |
| `button` | STRING | `buy`, `like`, `share` or `subscribe`. |
| `clicked_at` | TIMESTAMP_NTZ | When the click was recorded, to the millisecond. |

## Requirement

Return the clicks that are not accidental repeats.

- A click is a repeat if the same user clicked the **same button** less than 5
  seconds before it.
- "Before it" means the previous click on that button, whether or not that previous
  click was itself a repeat. Clicks at 0, 4, 8 and 12 seconds are one real click and
  three repeats, because each is 4 seconds after the one before.
- A click exactly 5.000 seconds after the previous one is not a repeat.
- If two clicks on the same button have the same timestamp, the one with the higher
  `click_id` is the repeat.
- Clicks on different buttons never affect each other.

## Expected output

One row per click that is kept.

| Column | Type | Description |
|---|---|---|
| `click_id` | BIGINT | |
| `user_id` | BIGINT | |
| `button` | STRING | |
| `clicked_at` | TIMESTAMP_NTZ | |

In [ ]:
clicks.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `clicks`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)